In [1]:
import os
import re
import pandas as pd
from PyPDF2 import PdfReader
from openai import OpenAI 
import time
import shutil

In [2]:
client = OpenAI(base_url="http://chatapi.littlewheat.com/v1", api_key="sk-ozguWsjyk4Of9ByNODJg7k9HC5kyoY6w3XCdIXcnEgMMO4Nm")

In [3]:
industries = ['化工', '传统能源', '金融', '医药', '消费', '房地产', '电动车新能源', '高端装备', '旅游', '半导体与信息技术']

In [4]:
def extract_info_from_filename(filename):

    name = os.path.splitext(filename)[0]
    

    pattern1 = r'\[(.*?)\](.*?)_(.*?)_(.*?)_(.*)'
    pattern2 = r'\[(.*?)\](.*?)__(.*)'
    

    match = re.match(pattern1, name)
    if match:
        date, title, agency, category, doc_number = match.groups()
        bulletin_number = ''
    else:
        match = re.match(pattern2, name)
        if match:
            date, title, bulletin_number = match.groups()
            agency = category = doc_number = ''
        else:
            date = title = agency = category = doc_number = bulletin_number = ''
    
    return date, title, agency, category, doc_number, bulletin_number


def determine_agency_level(agency):
    central_agencies = ['国务院']

    ministries = ['外交部', '国家发展和改革委员会', '教育部', '科学技术部', '工业和信息化部',
                  '国家民族事务委员会', '公安部', '国家安全部', '民政部', '司法部', '财政部',
                  '人力资源和社会保障部', '自然资源部', '生态环境部', '住房和城乡建设部',
                  '交通运输部', '水利部', '农业农村部', '商务部', '文化和旅游部',
                  '国家卫生健康委员会', '退役军人事务部', '应急管理部', '中国人民银行',
                  '审计署', '国家语言文字工作委员会', '国家航天局', '国家原子能机构',
                  '国家核安全局', '国家乡村振兴局', '国务院国有资产监督管理委员会',
                  '海关总署', '国家税务总局', '国家市场监督管理总局', '国家金融监督管理总局',
                  '中国证券监督管理委员会', '国家广播电视总局', '国家体育总局', '国家信访局',
                  '国家统计局', '国家知识产权局', '国家国际发展合作署', '国家医疗保障局',
                  '国家机关事务管理局', '国家认证认可监督管理委员会', '国家标准化管理委员会',
                  '国家新闻出版署', '国家版权局', '国家宗教事务局', '国务院台湾事务办公室',
                  '国家互联网信息办公室', '中国科学院', '中国社会科学院', '中国工程院',
                  '中国气象局', '国家粮食和物资储备局', '国家能源局', '国家数据局',
                  '国家国防科技工业局', '国家烟草专卖局', '国家移民管理局', '国家林业和草原局',
                  '国家铁路局', '中国民用航空局', '国家邮政局', '国家文物局', '国家中医药管理局',
                  '国家疾病预防控制局', '国家矿山安全监察局', '国家消防救援局', '国家外汇管理局',
                  '国家药品监督管理局', '国家公务员局', '国家档案局', '国家保密局',
                  '国家密码管理局', '国家电影局', '国家发展改革委', '发展改革委', '发改委', '科技部',
                  '国家民委', '安全部', '人力资源社会保障部', '人社部', '住房城乡建设部', '住建部',
                  '国家卫生健康委', '卫生健康委', '卫健委', '国家语委', '核安全局', '乡村振兴局', '国务院国资委',
                  '国资委', '税务总局', '市场监管总局', '金融监管总局', '中国证监会', '证监会', '广电总局',
                  '体育总局', '信访局', '统计局', '知识产权局', '国家医保局', '医保局', '国管局', '国家认监委',
                  '认监委', '国家标准委', '标准委', '新闻出版署', '版权局', '宗教局', '宗教事务局', '台办', '网信办',
                  '气象局', '粮食和储备局', '能源局', '数据局', '国防科工局', '国防科技工业局', '烟草专卖局',
                  '烟草局', '移民局', '移民管理局', '林业和草原局', '林草局', '铁路局', '民用航空局',
                  '民航局', '邮政局', '文物局', '中医药管理局', '中医药局', '疾控局', '疾病预防控制局', '矿山安监局',
                  '安监局', '外汇局', '外汇管理局', '消防救援局', '药监局', '药品监督管理局', '公务员局', '档案局',
                  '保密局', '密码局', '电影局']


    suffixes = ['办公厅', '办公室', '局', '处', '委员会', '中心', '总局', '署', '司']


    for ministry in ministries:
        if re.search(re.escape(ministry) + r'(\s*[一-龥]*(' + '|'.join(suffixes) + ')?)*$', agency):
            return '国务院部门'
        

    for central_agency in central_agencies:
        if re.search(r'^' + re.escape(central_agency) + r'(\s*[一-龥]*(' + '|'.join(suffixes) + ')?)*$', agency):
            return '国务院'

    if not agency:
        return ''


    return '其他'

In [5]:
def extract_text_from_pdf(pdf_path):
    print(f"正在提取PDF文本: {pdf_path}")
    text = ''
    reader = PdfReader(pdf_path)
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text
    return text

In [6]:
import re

def analyze_sentiment(text, industry):
    print(f"正在分析行业: {industry}")
    prompt = f"""请判断以下政策文件的内容对{industry}行业的投资策略的影响（正面、中性、负面三者中的一个），并给出置信度（百分比）。
只输出如下格式，不需要任何解释：
结果：正面或中性或负面 置信度：XX%
    
{text}"""
    
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.5,
    )
    
    response = completion.choices[0].message.content.strip()
    print(f"分析结果: {response}")
    
    match = re.search(r'结果[：:]*\s*[(（]?(正面|中性|负面)[）)]?\s*置信度[：:]*\s*(\d+)%', response)


    
    if match:
        sentiment = match.group(1)
        confidence = match.group(2) + "%"
    else:
        sentiment = "无法识别"
        confidence = "N/A"

    return sentiment, confidence


In [19]:
def main(pdf_directory, output_csv):
    file_count = int(input("请输入需要处理的PDF文件数量: "))
    processed_folder = os.path.join(pdf_directory, '已处理文件')
    if not os.path.exists(processed_folder):
        os.makedirs(processed_folder)

    df = pd.DataFrame(columns=['日期', '标题', '发文机关', '机关层级', '主题分类', '发文字号', '所属公报号码', '影响行业', '情感倾向', '情感置信度'])
    
    pdf_files = [f for f in os.listdir(pdf_directory) if f.endswith('.pdf')]
    pdf_files = pdf_files[:file_count]  

    for filename in pdf_files:
        print(f"正在处理文件: {filename}")
        date, title, agency, category, doc_number, bulletin_number = extract_info_from_filename(filename)
        agency_level = determine_agency_level(agency)
        pdf_path = os.path.join(pdf_directory, filename)
        text = extract_text_from_pdf(pdf_path)
        
        for industry in industries:
            sentiment, confidence = analyze_sentiment(text, industry)
            df = pd.concat([df, pd.DataFrame([{
                '日期': date,
                '标题': title,
                '发文机关': agency,
                '机关层级': agency_level,
                '主题分类': category,
                '发文字号': doc_number,
                '所属公报号码': bulletin_number,
                '影响行业': industry,
                '情感倾向': sentiment,
                '情感置信度': confidence
            }])], ignore_index=True)
            time.sleep(1)
        
        shutil.move(pdf_path, os.path.join(processed_folder, filename))
        print(f"文件已移动: {filename}")

    df.sort_values(by=['标题', '所属公报号码', '日期'], ascending=[True, False, False], inplace=True)
    df.drop_duplicates(subset=['标题', '影响行业'], keep='first', inplace=True)

    if os.path.exists(output_csv):
        df.to_csv(output_csv, index=False, mode='a', header=False) 
        print("处理完成，结果已追加。")
    else:
        df.to_csv(output_csv, index=False)
        print("处理完成，结果已保存。")




if __name__ == "__main__":
    pdf_directory = r'C:\Users\wrz\Desktop\FYPProject\数据源'
    output_csv = r'C:\Users\wrz\Desktop\FYPProject\output.csv'
    main(pdf_directory, output_csv)

请输入需要处理的PDF文件数量: 61
正在处理文件: [2024.8.20]重要商品和服务价格指数行为管理办法__2024年第23号国务院公报.pdf
正在提取PDF文本: C:\Users\wrz\Desktop\FYPProject\数据源\[2024.8.20]重要商品和服务价格指数行为管理办法__2024年第23号国务院公报.pdf
正在分析行业: 化工
分析结果: 结果：正面 置信度：75%
正在分析行业: 传统能源
分析结果: 结果：中性 置信度：70%
正在分析行业: 金融
分析结果: 结果：正面 置信度：75%
正在分析行业: 医药
分析结果: 结果：中性 置信度：70%
正在分析行业: 消费
分析结果: 结果：正面 置信度：75%
正在分析行业: 房地产
分析结果: 结果：中性 置信度：70%
正在分析行业: 电动车新能源
分析结果: 结果：中性 置信度：70%
正在分析行业: 高端装备
分析结果: 结果：中性 置信度：70%
正在分析行业: 旅游
分析结果: 结果：中性 置信度：70%
正在分析行业: 半导体与信息技术
分析结果: 结果：中性 置信度：70%
文件已移动: [2024.8.20]重要商品和服务价格指数行为管理办法__2024年第23号国务院公报.pdf
正在处理文件: [2024.8.24]关于印发《数字化绿色化协同转型发展实施指南》的通知_中央网信办秘书局 国家发展改革委办公厅 工业和信息化部办公厅 自然资源部办公厅 生态环境部办公厅 住房城乡建设部办公厅 交通运输部办公厅 农业农村部办公厅 市场监管总局办公厅 国家数据局综合司_工业、交通&信息产业（含电信）_.pdf
正在提取PDF文本: C:\Users\wrz\Desktop\FYPProject\数据源\[2024.8.24]关于印发《数字化绿色化协同转型发展实施指南》的通知_中央网信办秘书局 国家发展改革委办公厅 工业和信息化部办公厅 自然资源部办公厅 生态环境部办公厅 住房城乡建设部办公厅 交通运输部办公厅 农业农村部办公厅 市场监管总局办公厅 国家数据局综合司_工业、交通&信息产业（含电信）_.pdf
正在分析行业: 化工
分析结果: 结果：正面 置信度：75%
正在分析行业: 传统能源
分析结果: 结果：负面 置信度：75%
正在分析行业: 金融
分析结果: 

正在分析行业: 化工
分析结果: 结果：中性 置信度：70%
正在分析行业: 传统能源
分析结果: 结果：中性 置信度：70%
正在分析行业: 金融
分析结果: 结果：中性 置信度：70%
正在分析行业: 医药
分析结果: 结果：中性 置信度：70%
正在分析行业: 消费
分析结果: 结果：中性 置信度：70%
正在分析行业: 房地产
分析结果: 结果：中性 置信度：70%
正在分析行业: 电动车新能源
分析结果: 结果：中性 置信度：70%
正在分析行业: 高端装备
分析结果: 结果：中性 置信度：70%
正在分析行业: 旅游
分析结果: 结果：中性 置信度：70%
正在分析行业: 半导体与信息技术
分析结果: 结果：中性 置信度：70%
文件已移动: [2024.9.10]中共中央 国务院关于弘扬教育家精神加强新时代高素质专业化教师队伍建设的意见__2024年第25号国务院公报.pdf
正在处理文件: [2024.9.10]中共中央办公厅 国务院办公厅关于完善市场准入制度的意见__2024年第25号国务院公报.pdf
正在提取PDF文本: C:\Users\wrz\Desktop\FYPProject\数据源\[2024.9.10]中共中央办公厅 国务院办公厅关于完善市场准入制度的意见__2024年第25号国务院公报.pdf
正在分析行业: 化工
分析结果: 结果：正面 置信度：85%
正在分析行业: 传统能源
分析结果: 结果：中性 置信度：70%
正在分析行业: 金融
分析结果: 结果：正面 置信度：85%
正在分析行业: 医药
分析结果: 结果：正面 置信度：85%
正在分析行业: 消费
分析结果: 结果：正面 置信度：85%
正在分析行业: 房地产
分析结果: 结果：正面 置信度：75%
正在分析行业: 电动车新能源
分析结果: 结果：正面 置信度：85%
正在分析行业: 高端装备
分析结果: 结果：正面 置信度：85%
正在分析行业: 旅游
分析结果: 结果：正面 置信度：85%
正在分析行业: 半导体与信息技术
分析结果: 结果：正面 置信度：85%
文件已移动: [2024.9.10]中共中央办公厅 国务院办公厅关于完善市场准入制度的意见__2024年第25号国务院公报.pdf
正在处理文件: [2024.9.10]中华人民共和国出口货物原产地证书

正在分析行业: 旅游
分析结果: 结果：正面 置信度：85%
正在分析行业: 半导体与信息技术
分析结果: 结果：中性 置信度：70%
文件已移动: [2024.9.10]工业和信息化部 人力资源社会保障部 文化和旅游部关于推动工艺美术行业传承创新发　展的指导意见__2024年第25号国务院公报.pdf
正在处理文件: [2024.9.10]水利部关于印发《水利部野外科学观测研究站建设与运行管理暂行办法》的通知　　水利部野外科学观测研究站建设与运行管理暂行办法__2024年第25号国务院公报.pdf
正在提取PDF文本: C:\Users\wrz\Desktop\FYPProject\数据源\[2024.9.10]水利部关于印发《水利部野外科学观测研究站建设与运行管理暂行办法》的通知　　水利部野外科学观测研究站建设与运行管理暂行办法__2024年第25号国务院公报.pdf
正在分析行业: 化工
分析结果: 结果：中性 置信度：70%
正在分析行业: 传统能源
分析结果: 结果：中性 置信度：75%
正在分析行业: 金融
分析结果: 结果：中性 置信度：70%
正在分析行业: 医药
分析结果: 结果：中性 置信度：70%
正在分析行业: 消费
分析结果: 结果：中性 置信度：70%
正在分析行业: 房地产
分析结果: 结果：中性 置信度：70%
正在分析行业: 电动车新能源
分析结果: 结果：中性 置信度：70%
正在分析行业: 高端装备
分析结果: 结果：正面 置信度：85%
正在分析行业: 旅游
分析结果: 结果：中性 置信度：70%
正在分析行业: 半导体与信息技术
分析结果: 结果：中性 置信度：70%
文件已移动: [2024.9.10]水利部关于印发《水利部野外科学观测研究站建设与运行管理暂行办法》的通知　　水利部野外科学观测研究站建设与运行管理暂行办法__2024年第25号国务院公报.pdf
正在处理文件: [2024.9.10]财政部 科技部 工业和信息化部 金融监管总局关于实施支持科技创新专项担保计划的通知__2024年第25号国务院公报.pdf
正在提取PDF文本: C:\Users\wrz\Desktop\FYPProject\数据源\[2024.9.10]财政部 科技部 工业和信息化部 金融监管总局关于实施支持科技创新专项担保计划的通知__20

正在分析行业: 金融
分析结果: 结果：正面 置信度：75%
正在分析行业: 医药
分析结果: 结果：中性 置信度：70%
正在分析行业: 消费
分析结果: 结果：正面 置信度：85%
正在分析行业: 房地产
分析结果: 结果：中性 置信度：70%
正在分析行业: 电动车新能源
分析结果: 结果：中性 置信度：70%
正在分析行业: 高端装备
分析结果: 结果：正面 置信度：80%
正在分析行业: 旅游
分析结果: 结果：中性 置信度：75%
正在分析行业: 半导体与信息技术
分析结果: 结果：正面 置信度：85%
文件已移动: [2024.9.20]财政部关于印发《会计信息化工作规范》的通知__2024年第26号国务院公报.pdf
正在处理文件: [2024.9.20]金融监管总局关于加强和改进互联网财产保险业务监管有关事项的通知__2024年第26号国务院公报.pdf
正在提取PDF文本: C:\Users\wrz\Desktop\FYPProject\数据源\[2024.9.20]金融监管总局关于加强和改进互联网财产保险业务监管有关事项的通知__2024年第26号国务院公报.pdf
正在分析行业: 化工
分析结果: 结果：中性 置信度：75%
正在分析行业: 传统能源
分析结果: 结果：中性 置信度：75%
正在分析行业: 金融
分析结果: 结果：正面 置信度：85%
正在分析行业: 医药
分析结果: 结果：中性 置信度：70%
正在分析行业: 消费
分析结果: 结果：正面 置信度：75%
正在分析行业: 房地产
分析结果: 结果：中性 置信度：70%
正在分析行业: 电动车新能源
分析结果: 结果：中性 置信度：75%
正在分析行业: 高端装备
分析结果: 结果：中性 置信度：70%
正在分析行业: 旅游
分析结果: 结果：中性 置信度：75%
正在分析行业: 半导体与信息技术
分析结果: 结果：中性 置信度：70%
文件已移动: [2024.9.20]金融监管总局关于加强和改进互联网财产保险业务监管有关事项的通知__2024年第26号国务院公报.pdf
正在处理文件: [2024.9.20]金融监管总局关于印发《反保险欺诈工作办法》的通知　　反保险欺诈工作办法__2024年第26号国务院公报.pdf
正在提取PDF文本: C:\Users\wrz\Desk

分析结果: 结果：中性 置信度：70%
正在分析行业: 消费
分析结果: 结果：负面 置信度：85%
正在分析行业: 房地产
分析结果: 结果：负面 置信度：85%
正在分析行业: 电动车新能源
分析结果: 结果：负面 置信度：85%
正在分析行业: 高端装备
分析结果: 结果：负面 置信度：85%
正在分析行业: 旅游
分析结果: 结果：负面 置信度：85%
正在分析行业: 半导体与信息技术
分析结果: 结果：中性 置信度：70%
文件已移动: [2024.9.30]水利部 市场监管总局关于在黄河流域实行强制性用水定额管理的意见__2024年第27号国务院公报.pdf
正在处理文件: [2024.9.30]水利部关于印发《水利建设市场经营主体信用信息管理办法》的通知　　水利建设市场经营主体信用信息管理办法__2024年第27号国务院公报.pdf
正在提取PDF文本: C:\Users\wrz\Desktop\FYPProject\数据源\[2024.9.30]水利部关于印发《水利建设市场经营主体信用信息管理办法》的通知　　水利建设市场经营主体信用信息管理办法__2024年第27号国务院公报.pdf
正在分析行业: 化工
分析结果: 结果：中性 置信度：70%
正在分析行业: 传统能源
分析结果: 结果：中性 置信度：70%
正在分析行业: 金融
分析结果: 结果：正面 置信度：75%
正在分析行业: 医药
分析结果: 结果：中性 置信度：70%
正在分析行业: 消费
分析结果: 结果：中性 置信度：70%
正在分析行业: 房地产
分析结果: 结果：中性 置信度：70%
正在分析行业: 电动车新能源
分析结果: 结果：中性 置信度：70%
正在分析行业: 高端装备
分析结果: 结果：正面 置信度：80%
正在分析行业: 旅游
分析结果: 结果：中性 置信度：70%
正在分析行业: 半导体与信息技术
分析结果: 结果：中性 置信度：70%
文件已移动: [2024.9.30]水利部关于印发《水利建设市场经营主体信用信息管理办法》的通知　　水利建设市场经营主体信用信息管理办法__2024年第27号国务院公报.pdf
正在处理文件: [2024.9.30]财政部关于印发《会计软件基本功能和服务规范》的通知__2024年第27号国务院公报.pdf
正在提取PDF文本: C: